In [ ]:
import pandas as pd

In [ ]:
import sklearn
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.metrics import root_mean_squared_error
import mlflow

In [ ]:
mlflow.set_tracking_uri('sqlite:////workspaces/mlops_zoomcamp/mlflow.db')
mlflow.set_experiment('my_nyc_experiment')

In [ ]:
#!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet
#!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet

In [ ]:
def process_data(dv, categorical, numerical, target, data_path, val):
    data = pd.read_parquet(data_path)
    data['duration'] = data['tpep_dropoff_datetime'] - data ['tpep_pickup_datetime']
    data['duration'] = data['duration'].apply(lambda pd: pd.total_seconds()/60)
    data = data[(data['duration'] > 1) & (data ['duration'] <= 60)]
    data[categorical] = data[categorical].astype(str)
    data_dicts = data[categorical + numerical].to_dict(orient='records')
    if val:
        x_train = dv.transform(data_dicts)
    else:
        x_train = dv.fit_transform(data_dicts)
    y_train = data[target].values
    return x_train, y_train

In [ ]:
categorical = ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']
target = 'duration'
dv = DictVectorizer()
x_train, y_train = process_data(dv, categorical, numerical, target, "/workspaces/mlops_zoomcamp/01-intro/yellow_tripdata_2023-01.parquet", val=False)
x_val, y_val = process_data(dv, categorical, numerical, target, "/workspaces/mlops_zoomcamp/01-intro/yellow_tripdata_2023-02.parquet", True)

In [ ]:
lr = LinearRegression()
lr.fit(x_train, y_train)

In [ ]:
y_predict = lr.predict(x_train)
root_mean_squared_error(y_train, y_predict)

In [ ]:
y_predict_val = lr.predict(x_val)
root_mean_squared_error(y_val, y_predict_val)

In [ ]:
with mlflow.start_run():
    mlflow.set_tag("developer","rajesh")
    mlflow.log_param("train_data","yellow_tripdata_2023-01.parquet")
    mlflow.log_param("test_data","yellow_tripdata_2023-02.parquet")
    alpha = 0.01
    mlflow.log_param("alpha",alpha)
    lasso = Lasso(alpha)
    lasso.fit(x_train, y_train)
    y_pred = lasso.predict(x_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_param("rmse",rmse)